# Deep Learning in Practice -- Workshop Exercises

**3-hour hands-on workshop | FAMNIT AI Workshop -- Day 2**

| Hour | Topic | Exercise | Key Learning |
|------|-------|----------|--------------|
| 1 | Training a Deep Neural Network | Fashion-MNIST Classifier | Build & train DNNs with PyTorch, step by step |
| 2 | Embeddings + Chunking + Vector DB | LangChain + ChromaDB Pipeline | Embeddings, chunking strategies, semantic search & classification |
| 3 | Full Pipeline Comparison | DNN vs Embeddings | Decision guide: when to use which |

---

## Setup

1. Open this notebook in **Google Colab** or **Jupyter**
2. Run the install cell below
3. **Interactive app:** Visit the Streamlit app (link from instructor) for visual demos

> **Tip:** The Streamlit app at `streamlit-app/` lets you explore embeddings and ChromaDB interactively.
> Use it alongside this notebook for the best learning experience.

In [3]:
# Install dependencies (run this cell first)
!pip install torch torchvision langchain langchain-community langchain-huggingface \
    chromadb sentence-transformers scikit-learn matplotlib numpy pandas seaborn plotly -q

ERROR: Could not find a version that satisfies the requirement torch (from versions: none)

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
ERROR: No matching distribution found for torch


In [ ]:
# Core imports and configuration
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

# Dark theme with Google-style colors
plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# Google colors for consistent styling
BLUE = '#4285f4'
RED = '#ea4335'
GREEN = '#34a853'
YELLOW = '#fbbc04'

import torch
import torch.nn as nn
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
print('All packages loaded successfully!')

---

# Quick Review: Types of Machine Learning

Before we dive into deep learning, let's quickly review the three main types of machine learning.

| Type | How it learns | Data needed | Example |
|------|--------------|-------------|----------|
| **Supervised Learning** | Learning with a teacher | Labeled data (input + correct answer) | "This image is a cat" --> learns to recognize cats |
| **Unsupervised Learning** | Finding patterns on your own | Unlabeled data (just inputs) | Grouping customers by shopping behavior |
| **Reinforcement Learning** | Learning by trial and error | Rewards and penalties | A robot learning to walk |

### Simple analogies

- **Supervised Learning** = studying with a textbook that has answer keys. You check your answers, learn from mistakes, and improve.
- **Unsupervised Learning** = sorting a pile of photos without anyone telling you the categories. You notice patterns and group similar ones together.
- **Reinforcement Learning** = like training a dog -- it tries actions, gets treats (rewards) or corrections (penalties), and learns which behaviors lead to treats. The dog (agent) explores the environment, receives feedback, and develops a strategy (policy).

**Today we focus on Supervised Learning** -- we will train a neural network on labeled images (Hour 1) and use pre-trained embeddings for classification (Hour 2).

---

# HOUR 1: Training a Deep Neural Network

**Goal:** Build, train, and evaluate a feedforward neural network from scratch using PyTorch.

We will break this into 15 small steps so you understand every piece of the puzzle.

---

## Step 1: Import & Setup

Let's import the libraries we need. Here is what each one does:

| Library | Purpose |
|---------|----------|
| `torch` | The deep learning framework (like TensorFlow, but more Pythonic) |
| `torch.nn` | Building blocks for neural networks (layers, activation functions) |
| `torchvision` | Ready-made datasets and image transforms |
| `DataLoader` | Feeds data to the network in small batches |
| `matplotlib` | Plotting and visualization |

In [2]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

print('PyTorch version:', torch.__version__)
print('Device:', device)
print()
print('Ready! Let\'s load some data.')

ModuleNotFoundError: No module named 'torchvision'

## Step 2: Load the Data (Fashion-MNIST)

**Fashion-MNIST** is a dataset of 70,000 grayscale images of clothing items. It was created as a more challenging replacement for the classic MNIST handwritten digits dataset.

- **60,000** training images (what the network learns from)
- **10,000** test images (what we evaluate on -- the network never sees these during training)
- **10 classes:** T-shirt, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Ankle boot
- Each image is **28 x 28 pixels**, grayscale (1 channel)

The `transforms.ToTensor()` converts each image from a PIL image (0-255 integers) to a PyTorch tensor (0.0-1.0 floats).

In [ ]:
# ToTensor() converts pixel values from 0-255 to 0.0-1.0
transform = transforms.Compose([transforms.ToTensor()])

# Download Fashion-MNIST (this will only download once)
train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

# DataLoader handles batching and shuffling for us
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Human-readable class names
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print(f'Training images: {len(train_dataset):,}')
print(f'Test images:     {len(test_dataset):,}')
print(f'Classes:         {len(class_names)}')
print(f'Image shape:     {train_dataset[0][0].shape}  (channels, height, width)')
print(f'Pixel range:     [{train_dataset[0][0].min():.1f}, {train_dataset[0][0].max():.1f}]')

## Step 3: Visualize the Data

Always look at your data before building a model! Each image is a 28x28 grid of pixel values, where 0.0 = black and 1.0 = white.

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    image, label = train_dataset[i]
    ax.imshow(image.squeeze(), cmap='gray')
    ax.set_title(class_names[label], fontsize=9)
    ax.axis('off')
plt.suptitle('Fashion-MNIST Samples', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Let's also look at one image as raw numbers
print('\nOne image as a grid of numbers (first 10x10 corner):')
sample_image = train_dataset[0][0].squeeze().numpy()
print(np.round(sample_image[:10, :10], 2))
print(f'\nThese {28*28} = 784 numbers are all the network sees!')

## Step 4: What is a Neuron?

A **neuron** is the simplest building block of a neural network. It does just one thing:

```
output = input * weight + bias
```

That's it! A neuron takes an input, multiplies it by a **weight** (how important is this input?), adds a **bias** (a baseline offset), and produces an output.

Think of it like a volume knob (weight) with a starting position (bias).

In [ ]:
# A single neuron: output = input * weight + bias
input_value = 0.5
weight = 0.8
bias = 0.1
output = input_value * weight + bias
print(f'Input: {input_value} --> Output: {output}')
print(f'Calculation: {input_value} * {weight} + {bias} = {output}')
print()

# With multiple inputs (more realistic)
inputs = np.array([0.5, 0.3, 0.9])   # 3 input values
weights = np.array([0.8, -0.2, 0.5])  # 3 weights (one per input)
bias = 0.1
output = np.dot(inputs, weights) + bias  # weighted sum + bias
print(f'Multiple inputs: {inputs}')
print(f'Weights:         {weights}')
print(f'Weighted sum:    {inputs[0]}*{weights[0]} + {inputs[1]}*{weights[1]} + {inputs[2]}*{weights[2]} + {bias} = {output:.2f}')
print()
print('This is ALL a neuron does: weighted sum + bias.')
print('Training = finding the best weights and biases!')

## Step 5: What is a Layer?

A **layer** is a group of neurons working in parallel. Each neuron in the layer:
- Receives ALL the same inputs
- Has its OWN set of weights and bias
- Produces its OWN output

In PyTorch, `nn.Linear(784, 256)` creates a layer with:
- 784 inputs (one per pixel in our flattened 28x28 image)
- 256 neurons (each learning different patterns)
- 784 * 256 + 256 = 200,960 total parameters (weights + biases)

In [ ]:
# Create a single layer: 784 inputs, 256 neurons
layer = nn.Linear(784, 256)

# How many parameters does this layer have?
n_weights = 784 * 256       # each neuron has 784 weights
n_biases = 256              # each neuron has 1 bias
n_total = n_weights + n_biases

print(f'Layer: nn.Linear(784, 256)')
print(f'  Weights: 784 inputs x 256 neurons = {n_weights:,} weights')
print(f'  Biases:  256 neurons x 1 bias each = {n_biases:,} biases')
print(f'  Total parameters: {n_total:,}')
print()

# Pass a fake image through the layer
fake_image = torch.randn(1, 784)  # 1 image, 784 pixels (flattened)
layer_output = layer(fake_image)
print(f'Input shape:  {fake_image.shape}  (1 image, 784 pixels)')
print(f'Output shape: {layer_output.shape}  (1 image, 256 neuron outputs)')
print()
print('Each of the 256 neurons learned something different about the input!')

## Step 6: Build the Full Network

Now we **stack layers** to create a deep neural network. Between layers, we add:
- **ReLU** (activation function): Adds non-linearity. Without it, stacking layers would be no better than a single layer. `ReLU(x) = max(0, x)` -- simply "turn off" negative values.
- **Dropout**: Randomly "turns off" some neurons during training to prevent overfitting (like studying by randomly covering parts of your notes).

Our architecture:
```
Image (28x28) --> Flatten (784) --> Layer1 (256) --> ReLU --> Dropout
              --> Layer2 (128) --> ReLU --> Dropout --> Layer3 (10) --> Predictions
```

In [ ]:
class FashionNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),              # 28x28 image --> 784-dim vector
            nn.Linear(784, 256),       # Layer 1: 784 inputs --> 256 neurons
            nn.ReLU(),                 # Activation: turn off negatives
            nn.Dropout(0.3),           # Drop 30% of neurons randomly (prevents overfitting)
            nn.Linear(256, 128),       # Layer 2: 256 --> 128 neurons
            nn.ReLU(),                 # Activation again
            nn.Dropout(0.2),           # Drop 20% of neurons randomly
            nn.Linear(128, 10),        # Output: 128 --> 10 classes
        )

    def forward(self, x):
        return self.network(x)

model = FashionNet().to(device)

# Count parameters
n_params = sum(p.numel() for p in model.parameters())
print(f'Model has {n_params:,} trainable parameters')
print()
print('Architecture:')
print(model)
print()
print(f'These {n_params:,} numbers (weights + biases) are what the network LEARNS.')
print('Right now they are random. Training will find the right values.')

## Step 7: The Forward Pass

The **forward pass** is simply pushing data through the network from input to output. Right now the weights are random, so the predictions will be garbage -- but let's see the mechanics.

In [ ]:
# Grab one batch of images
images, labels = next(iter(train_loader))
images = images.to(device)

# Forward pass: push images through the network
model.eval()  # Turn off dropout for this demo
with torch.no_grad():
    outputs = model(images)

print(f'Input shape:  {images.shape}   (64 images, 1 channel, 28x28)')
print(f'Output shape: {outputs.shape}    (64 images, 10 class scores)')
print()

# Look at the raw output for the first image
print('Raw scores for the first image (one per class):')
for i, (name, score) in enumerate(zip(class_names, outputs[0].cpu().numpy())):
    print(f'  {name:<12} {score:>7.3f}')

predicted_class = outputs[0].argmax().item()
actual_class = labels[0].item()
print(f'\nPredicted: {class_names[predicted_class]}')
print(f'Actual:    {class_names[actual_class]}')
print(f'\nThe prediction is {"CORRECT!" if predicted_class == actual_class else "WRONG (expected -- weights are random!)"}')

## Step 8: The Loss Function -- "How wrong are we?"

The **loss function** measures how far the network's predictions are from the correct answers.

We use **CrossEntropyLoss**, which:
1. Converts raw scores to probabilities (via softmax)
2. Measures how "surprised" the model is by the correct answer
3. Returns a single number: **lower = better**

Think of it as a "wrongness score" -- our goal is to minimize it.

In [ ]:
criterion = nn.CrossEntropyLoss()

# Compute loss on our batch
model.eval()
with torch.no_grad():
    outputs = model(images)
    loss = criterion(outputs, labels.to(device))

print(f'Loss: {loss.item():.4f}')
print()
print(f'For random guessing among 10 classes, expected loss is about {-np.log(1/10):.4f}')
print(f'Our loss is {loss.item():.4f} -- close to random, as expected with random weights.')
print()
print('Goal: drive this loss DOWN through training!')

## Step 9: Backpropagation -- "Blame assignment"

**Backpropagation** answers the question: *which weights caused the error?*

It works backward through the network, computing **gradients** -- the direction and magnitude each weight should change to reduce the loss.

- If a weight has a **positive gradient**: decreasing it will reduce the loss
- If a weight has a **negative gradient**: increasing it will reduce the loss
- If a weight has a **gradient near zero**: it doesn't matter much for this prediction

PyTorch does this automatically with `loss.backward()` -- you don't need to compute derivatives by hand!

In [ ]:
# Forward pass
model.train()
outputs = model(images)
loss = criterion(outputs, labels.to(device))

# Before backward pass -- no gradients yet
first_layer = list(model.parameters())[0]
print(f'Gradients before backward: {first_layer.grad}')

# Backward pass -- compute all gradients
loss.backward()

# After backward pass -- gradients computed!
print(f'Gradients after backward:  shape={first_layer.grad.shape}, mean={first_layer.grad.mean():.6f}')
print()
print(f'PyTorch computed gradients for all {n_params:,} parameters automatically!')
print('Each gradient says: "change this weight by this much to reduce the loss."')

## Step 10: The Optimizer -- "Adjust the weights"

The **optimizer** uses the gradients from backpropagation to actually update the weights.

We use **Adam** (Adaptive Moment Estimation), one of the most popular optimizers. It is like a smart student who:
- Remembers which direction it has been going (momentum)
- Adjusts its step size for each parameter individually (adaptive learning rate)

The **learning rate** (lr) controls how big each step is:
- Too high: overshoots and diverges
- Too low: learns too slowly
- Just right: converges smoothly

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Show weight before update
weight_before = list(model.parameters())[0].data[0, 0].item()
print(f'Weight before update: {weight_before:.6f}')

# Update weights using gradients!
optimizer.step()

# Show weight after update
weight_after = list(model.parameters())[0].data[0, 0].item()
print(f'Weight after update:  {weight_after:.6f}')
print(f'Change:               {weight_after - weight_before:.6f}')
print()
print('The optimizer nudged all weights slightly to reduce the loss.')
print('Repeat this thousands of times = training!')

## Step 11: One Complete Training Step

Let's combine all four pieces into **one complete training step**. This is the heartbeat of deep learning:

```
1. Forward pass   --> push data through the network
2. Compute loss   --> measure how wrong we are
3. Backward pass  --> compute gradients (blame assignment)
4. Optimizer step --> update weights to reduce loss
```

In [ ]:
# Re-create the model and optimizer (fresh start for the full training)
model = FashionNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# ONE training step on ONE batch
model.train()
images, labels = next(iter(train_loader))
images, labels = images.to(device), labels.to(device)

# Step 1: Forward pass
outputs = model(images)

# Step 2: Compute loss
loss = criterion(outputs, labels)
print(f'Loss BEFORE update: {loss.item():.4f}')

# Step 3: Backward pass (compute gradients)
optimizer.zero_grad()   # Clear old gradients first!
loss.backward()         # Compute new gradients

# Step 4: Update weights
optimizer.step()        # Apply gradients to weights

# Check: did the loss go down?
with torch.no_grad():
    new_outputs = model(images)
    new_loss = criterion(new_outputs, labels)
print(f'Loss AFTER update:  {new_loss.item():.4f}')
print(f'Improvement:        {loss.item() - new_loss.item():.4f}')
print()
print('One step done! The loss went down. Now repeat this for all batches and all epochs.')

## Step 12: Full Training Loop

Now we put it all together. One **epoch** = one pass through all 60,000 training images.

We train for **10 epochs**, and after each epoch we evaluate on the test set to see how well the model generalizes to unseen data.

In [ ]:
# Fresh model for the real training
model = FashionNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 10
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print(f'Training for {EPOCHS} epochs on {len(train_dataset):,} images...\n')
print(f'{"Epoch":>6} {"Train Loss":>11} {"Train Acc":>10} {"Val Loss":>10} {"Val Acc":>9}')
print('-' * 50)

for epoch in range(EPOCHS):
    # --- Training phase ---
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()          # 1. Clear old gradients
        outputs = model(images)        # 2. Forward pass
        loss = criterion(outputs, labels)  # 3. Compute loss
        loss.backward()                # 4. Backward pass (gradients)
        optimizer.step()               # 5. Update weights

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total

    # --- Validation phase ---
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():  # No gradients needed for evaluation
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            val_correct += predicted.eq(labels).sum().item()
            val_total += labels.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(f'{epoch+1:>6d} {train_loss:>11.4f} {train_acc:>10.1%} {val_loss:>10.4f} {val_acc:>9.1%}')

print(f'\nFinal test accuracy: {val_acc:.1%}')

## Step 13: Visualize Training

Plotting the **loss curve** and **accuracy curve** helps us understand how training went:
- If training loss goes down but validation loss goes up --> **overfitting** (memorizing, not learning)
- If both go down together --> **healthy training**

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
ax1.plot(history['train_loss'], 'o-', color=RED, label='Train Loss', linewidth=2)
ax1.plot(history['val_loss'], 'o-', color=BLUE, label='Val Loss', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Loss Curve', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(alpha=0.3)

# Accuracy curve
ax2.plot([a * 100 for a in history['train_acc']], 'o-', color=RED, label='Train Acc', linewidth=2)
ax2.plot([a * 100 for a in history['val_acc']], 'o-', color=BLUE, label='Val Acc', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Accuracy Curve', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print('If the training and validation curves stay close --> healthy learning!')
print('If they diverge --> the model is overfitting (memorizing the training data).')

## Step 14: Confusion Matrix and Evaluation

The **confusion matrix** shows exactly which classes the model confuses. The diagonal shows correct predictions; off-diagonal cells show mistakes.

**Think about it:** Which classes do you think will be most confused? (Hint: Shirt vs T-shirt!)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

# Collect all predictions
all_preds, all_labels = [], []
model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title(f'Confusion Matrix (Accuracy: {val_acc:.1%})', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(classification_report(all_labels, all_preds, target_names=class_names))

## Step 15: Reflection -- Hour 1

| What You Did | What You Learned |
|-------------|------------------|
| Explored a single neuron | `output = input * weight + bias` -- that's it! |
| Built layers from neurons | Many neurons in parallel = a layer |
| Stacked layers into a network | Deep = multiple layers, each learning more abstract features |
| Ran the forward pass | Data flows through the network: input --> output |
| Computed the loss | CrossEntropyLoss measures "how wrong are we?" |
| Ran backpropagation | Gradients tell us which weights caused the error |
| Used the optimizer | Adam adjusts weights to reduce the loss |
| Trained for 10 epochs | Repeat the loop thousands of times to learn |
| Plotted loss/accuracy | Monitor training health and detect overfitting |
| Used a confusion matrix | See which classes the model confuses |

**Key insight:** The training loop is universal -- the same 4 steps (forward, loss, backward, step) are used for everything from simple classifiers to GPT.

**Think about it:** We trained from scratch on 60,000 images. What if we only had 100 images? Would this approach still work? (Hint: probably not well -- that's where Hour 2 comes in!)

---

# HOUR 2: Embeddings + Chunking + Vector Database

**Goal:** Use **pre-trained embeddings**, learn **chunking strategies** for documents, and build a **semantic search & classification** pipeline with **LangChain** and **ChromaDB**.

### What's different from Hour 1?

| Hour 1 (DNN) | Hour 2 (Embeddings) |
|---|---|
| Train from scratch | Use a pre-trained model |
| Learn features from data | Features come for free |
| Need lots of labeled data | Works with small datasets |
| Hours to train | Seconds to embed |
| Custom to your dataset | General-purpose understanding |

---

## Step 1: What are Embeddings?

Think of embeddings as **GPS coordinates for meaning**.

Just like GPS coordinates tell you how close two **places** are, embedding coordinates tell you how close two **meanings** are.

```
"king"    --> [0.2, 0.8, 0.1, ...]   (384 numbers)
"queen"   --> [0.3, 0.7, 0.1, ...]   (nearby in embedding space!)
"banana"  --> [0.9, 0.1, 0.6, ...]   (far away -- different meaning)
```

The embedding model (`all-MiniLM-L6-v2`) was trained on millions of text pairs to learn these "meaning coordinates." We get to use its knowledge for free!

### What is LangChain?
LangChain is a framework that provides a clean, unified interface to embedding models, LLMs, and AI tools.

### What is ChromaDB?
ChromaDB is a **vector database** -- it stores documents as vectors and lets you search by **meaning**, not keywords.

## Step 2: Create Your First Embedding

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

# Load the pre-trained embedding model via LangChain
# all-MiniLM-L6-v2: a small but powerful model (22M parameters, 384-dim output)
embeddings_model = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')

# Embed a single sentence
example_text = "Machine learning is a subset of artificial intelligence"
vector = embeddings_model.embed_query(example_text)

print(f'Text: "{example_text}"')
print(f'Embedding dimensions: {len(vector)}')
print(f'First 10 values: {[round(v, 4) for v in vector[:10]]}')
print()
print(f'This vector of {len(vector)} numbers captures the MEANING of the text!')
print('Texts with similar meanings will have similar vectors.')

ModuleNotFoundError: No module named 'langchain_huggingface'

## Step 3: Compare Texts by Meaning

Let's embed several sentences and see how **cosine similarity** reveals semantic relationships. Cosine similarity ranges from -1 to 1:
- **1.0** = identical meaning
- **0.0** = completely unrelated
- **-1.0** = opposite meaning

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

texts = [
    "I love machine learning",
    "Artificial intelligence is fascinating",
    "Deep learning uses neural networks",
    "The weather is sunny today",
    "I need to buy groceries",
    "Neural networks learn from data",
]

# Embed all texts at once
vectors = np.array(embeddings_model.embed_documents(texts))
print(f'Embedded {len(texts)} texts into {vectors.shape[1]}-dimensional vectors')

# Compute cosine similarity matrix
sim_matrix = cosine_similarity(vectors)

plt.figure(figsize=(10, 8))
sns.heatmap(sim_matrix, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=[t[:30] for t in texts],
            yticklabels=[t[:30] for t in texts])
plt.title('Cosine Similarity Between Texts', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Notice: AI-related texts have HIGH similarity to each other,')
print('but LOW similarity to unrelated texts (weather, groceries).')

## Step 4: Why Chunking Matters

Real documents (papers, books, web pages) are often **too long** to embed as a single piece. Embedding models have a maximum token limit (typically 256-512 tokens for `all-MiniLM-L6-v2`).

**Chunking** = splitting a long document into smaller pieces before embedding.

### The chunking trade-off:

| Chunk size | Pros | Cons |
|-----------|------|------|
| **Too small** (1-2 sentences) | Precise retrieval | Loses context |
| **Too large** (whole pages) | Keeps context | Imprecise, may exceed token limits |
| **Just right** (100-500 chars) | Good balance | Requires experimentation |

Let's explore three chunking strategies and compare them!

In [ ]:
# A sample long document to chunk
long_document = """Artificial Intelligence (AI) has transformed numerous industries in recent years.
Machine learning, a subset of AI, enables computers to learn from data without being explicitly programmed.
Deep learning, which uses neural networks with many layers, has achieved remarkable results in image recognition, natural language processing, and game playing.

One of the most significant breakthroughs in AI has been the development of transformer models.
Transformers use a mechanism called attention to process sequences of data in parallel, rather than sequentially.
This architecture was introduced in the famous paper "Attention Is All You Need" in 2017.
Since then, transformer-based models like BERT, GPT, and T5 have set new benchmarks across many NLP tasks.

Embeddings are another crucial concept in modern AI.
An embedding is a dense vector representation of data that captures semantic meaning.
Word embeddings like Word2Vec and GloVe map words to vectors where similar words are close together.
Sentence embeddings extend this idea to entire sentences, enabling semantic search and text classification.

Vector databases have emerged as essential tools for working with embeddings at scale.
They store vectors and support fast similarity search using algorithms like HNSW (Hierarchical Navigable Small World).
Popular vector databases include ChromaDB, Pinecone, Weaviate, and Milvus.
These databases power applications like semantic search, recommendation systems, and retrieval-augmented generation (RAG)."""

print(f'Document length: {len(long_document)} characters')
print(f'Word count: ~{len(long_document.split())} words')
print()
print('This is too long to embed as a single chunk.')
print('Let\'s split it using three different strategies...')

## Step 5: Chunking Strategy 1 -- Fixed-Size

The simplest approach: split the text into chunks of a fixed character count, with **overlap** to avoid cutting in the middle of ideas.

**Overlap** ensures that information at chunk boundaries is not lost -- each chunk shares some text with the next.

In [ ]:
def fixed_size_chunk(text, chunk_size=200, overlap=50):
    """Split text into fixed-size chunks with overlap."""
    chunks = []
    for i in range(0, len(text), chunk_size - overlap):
        chunk = text[i:i + chunk_size]
        if chunk.strip():  # Skip empty chunks
            chunks.append(chunk.strip())
    return chunks

fixed_chunks = fixed_size_chunk(long_document, chunk_size=200, overlap=50)

print(f'Fixed-size chunking: {len(fixed_chunks)} chunks (size=200, overlap=50)')
print('=' * 70)
for i, chunk in enumerate(fixed_chunks):
    print(f'\nChunk {i+1} ({len(chunk)} chars):')
    print(f'  "{chunk[:80]}..."')

## Step 6: Chunking Strategy 2 -- Sentence-Based

A smarter approach: split by **sentence boundaries**. This respects the natural structure of text -- each chunk is a complete thought.

We use a simple regex to split on periods, question marks, and exclamation marks followed by whitespace.

In [ ]:
import re

def sentence_chunk(text):
    """Split text into individual sentences."""
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in sentences if s.strip()]

sentence_chunks = sentence_chunk(long_document)

print(f'Sentence-based chunking: {len(sentence_chunks)} chunks')
print('=' * 70)
for i, chunk in enumerate(sentence_chunks):
    print(f'\nSentence {i+1} ({len(chunk)} chars):')
    print(f'  "{chunk[:80]}..."' if len(chunk) > 80 else f'  "{chunk}"')

## Step 7: Chunking Strategy 3 -- Recursive Character Text Splitting (LangChain)

LangChain's `RecursiveCharacterTextSplitter` is the **recommended approach** for most use cases. It:

1. Tries to split on paragraph breaks (`\n\n`) first
2. If chunks are still too large, splits on line breaks (`\n`)
3. Then on sentences (`. `)
4. Then on spaces (` `)
5. Finally on characters (`""`) as a last resort

This hierarchy tries to keep semantically related text together as much as possible.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""],  # Try these in order
)

langchain_chunks = splitter.split_text(long_document)

print(f'LangChain recursive chunking: {len(langchain_chunks)} chunks (size=200, overlap=50)')
print('=' * 70)
for i, chunk in enumerate(langchain_chunks):
    print(f'\nChunk {i+1} ({len(chunk)} chars):')
    print(f'  "{chunk[:80]}..."' if len(chunk) > 80 else f'  "{chunk}"')

## Step 8: Compare Chunking Strategies

Let's compare all three strategies side by side.

In [ ]:
# Summary comparison
strategies = {
    'Fixed-Size': fixed_chunks,
    'Sentence-Based': sentence_chunks,
    'LangChain Recursive': langchain_chunks,
}

print(f'{"Strategy":<22} {"Chunks":<8} {"Avg Size":<10} {"Min Size":<10} {"Max Size":<10}')
print('-' * 60)
for name, chunks in strategies.items():
    sizes = [len(c) for c in chunks]
    print(f'{name:<22} {len(chunks):<8} {np.mean(sizes):<10.0f} {min(sizes):<10} {max(sizes):<10}')

print()

# Visualize chunk sizes
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors_list = [BLUE, GREEN, YELLOW]

for ax, (name, chunks), color in zip(axes, strategies.items(), colors_list):
    sizes = [len(c) for c in chunks]
    ax.bar(range(len(sizes)), sizes, color=color, edgecolor='white', linewidth=1)
    ax.set_title(f'{name}\n({len(chunks)} chunks)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Chunk #')
    ax.set_ylabel('Characters')
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Chunk Size Comparison Across Strategies', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print()
print('Comparison of chunking strategies:')
print()
comparison_data = {
    'Strategy': ['Fixed-Size', 'Sentence-Based', 'LangChain Recursive'],
    'Pros': [
        'Simple, predictable chunk sizes',
        'Each chunk is a complete thought',
        'Smart splitting, respects structure',
    ],
    'Cons': [
        'May cut in the middle of sentences',
        'Uneven sizes, some chunks too small',
        'Slightly more complex to configure',
    ],
    'Best for': [
        'Uniform data, simple use cases',
        'Well-structured text (articles, papers)',
        'General purpose (recommended default)',
    ],
}
comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))

## Step 9: Store Chunks in ChromaDB

Now let's store our chunks in a **vector database** (ChromaDB). This allows us to search across chunks by meaning.

When you call `Chroma.from_texts()`, LangChain:
1. Embeds each chunk using the embedding model
2. Stores the vector + original text in ChromaDB
3. Builds an index for fast similarity search

In [ ]:
from langchain_community.vectorstores import Chroma

# Let's also add more documents for a richer knowledge base
documents = [
    "Black holes are regions of spacetime where gravity is so strong that nothing can escape.",
    "DNA carries genetic instructions for the development of all living organisms.",
    "Machine learning algorithms improve through experience without being explicitly programmed.",
    "Quantum computers use qubits that can exist in multiple states simultaneously.",
    "Climate change is causing global temperatures to rise and weather patterns to shift.",
    "Antibiotics are medicines that fight bacterial infections in humans and animals.",
    "The Large Hadron Collider accelerates particles to near the speed of light.",
    "CRISPR technology allows scientists to edit genes with unprecedented precision.",
    "Neural networks are inspired by the structure of the human brain.",
    "Photosynthesis converts sunlight into chemical energy in plants.",
]

# Combine our AI-related chunks with the science documents
all_docs = langchain_chunks + documents

# Store everything in ChromaDB
vectorstore = Chroma.from_texts(
    texts=all_docs,
    embedding=embeddings_model,
    collection_name="knowledge_base",
)

print(f'Stored {len(all_docs)} documents/chunks in ChromaDB!')
print(f'  - {len(langchain_chunks)} chunks from the AI document')
print(f'  - {len(documents)} standalone science documents')
print(f'\nEach was automatically embedded into a {len(vector)}-dim vector.')
print('Ready for semantic search!')

## Step 10: Semantic Search Across Chunks

Now the magic happens -- we can search our knowledge base by **meaning**, not keywords. ChromaDB finds the documents whose embedding vectors are closest to the query's embedding.

In [ ]:
queries = [
    "How do computers learn?",
    "What happens inside a star?",
    "How do plants make food?",
    "Gene editing technology",
    "What are transformer models?",
    "How do vector databases work?",
]

for query in queries:
    print(f'\nQuery: "{query}"')
    print('-' * 60)
    results = vectorstore.similarity_search_with_relevance_scores(query, k=2)
    for i, (doc, score) in enumerate(results, 1):
        text_preview = doc.page_content[:80].replace('\n', ' ')
        print(f'  {i}. [relevance: {score:.2f}] {text_preview}...')

print('\n')
print('Notice: The search finds relevant documents even when the exact words differ!')
print('"How do computers learn?" finds documents about machine learning and neural networks.')
print('"What are transformer models?" finds the right chunk from our long document!')

## Step 11: Classification Pipeline (Embeddings + sklearn)

Now let's use embeddings for a real **classification task**. The pipeline is simple:

```
Documents --> LangChain Embeddings --> Vectors (384-dim) --> sklearn Classifier --> Predictions
```

We will use the **20 Newsgroups** dataset (real newsgroup posts) and compare multiple classifiers.

**Think about it:** We are NOT training a neural network here. We use a pre-trained model to convert text to vectors, then feed those vectors to simple classifiers. Much faster!

In [ ]:
from sklearn.datasets import fetch_20newsgroups

# Pick 4 distinct categories
categories = ['sci.space', 'comp.graphics', 'rec.sport.baseball', 'talk.politics.guns']

train_data = fetch_20newsgroups(
    subset='train', categories=categories,
    remove=('headers', 'footers', 'quotes')  # Remove metadata to avoid cheating
)
test_data = fetch_20newsgroups(
    subset='test', categories=categories,
    remove=('headers', 'footers', 'quotes')
)

print(f'Training documents: {len(train_data.data)}')
print(f'Test documents:     {len(test_data.data)}')
print(f'Categories: {train_data.target_names}')
print()
print('Sample document (first 200 chars):')
print(f'  Category: {categories[train_data.target[0]]}')
print(f'  Text: "{train_data.data[0][:200]}..."')

In [ ]:
# Embed all documents using LangChain
print('Encoding training texts with LangChain...')
t0 = time.time()
X_train_emb = np.array(embeddings_model.embed_documents(train_data.data))
t_train = time.time() - t0

print('Encoding test texts...')
t0 = time.time()
X_test_emb = np.array(embeddings_model.embed_documents(test_data.data))
t_test = time.time() - t0

y_train = train_data.target
y_test = test_data.target

print(f'\nTraining embeddings shape: {X_train_emb.shape}')
print(f'Test embeddings shape:     {X_test_emb.shape}')
print(f'Encoding time: {t_train + t_test:.1f}s')
print(f'\nEach document is now a {X_train_emb.shape[1]}-dimensional vector!')

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# Compare three classifiers
classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'SVM (RBF)': SVC(kernel='rbf', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
}

emb_results = {}
print(f'{"Classifier":<25} {"Accuracy":>10} {"Time":>8}')
print('-' * 45)

for name, clf in classifiers.items():
    t0 = time.time()
    clf.fit(X_train_emb, y_train)
    preds = clf.predict(X_test_emb)
    elapsed = time.time() - t0
    acc = accuracy_score(y_test, preds)
    emb_results[name] = {'accuracy': acc, 'time': elapsed, 'preds': preds}
    print(f'{name:<25} {acc:>10.1%} {elapsed:>7.2f}s')

print()
print('All three classifiers achieve high accuracy because the embeddings')
print('already capture the meaning of each document!')

In [ ]:
# Visualize classifier comparison
names = list(emb_results.keys())
accs = [emb_results[n]['accuracy'] * 100 for n in names]
colors_bar = [BLUE, RED, GREEN]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(names, accs, color=colors_bar, edgecolor='white', linewidth=1.5, width=0.5)
for bar, val in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width() / 2., bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', fontweight='bold', fontsize=13)
ax.set_ylabel('Accuracy (%)', fontsize=13)
ax.set_title('Classifier Comparison on LangChain Embeddings', fontsize=14, fontweight='bold')
ax.set_ylim(0, 105)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Step 12: t-SNE Visualization of Embedding Space

**t-SNE** (t-distributed Stochastic Neighbor Embedding) lets us visualize 384-dimensional vectors in 2D. If the embeddings are good, documents from the same category should form **clusters**.

In [ ]:
from sklearn.manifold import TSNE

print('Computing t-SNE projection (this may take a minute)...')
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(X_train_emb)

colors_scatter = [BLUE, RED, GREEN, YELLOW]

plt.figure(figsize=(10, 8))
for i, cat in enumerate(categories):
    mask = y_train == i
    plt.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                c=colors_scatter[i], label=cat, s=20, alpha=0.6)
plt.legend(fontsize=11, markerscale=3)
plt.title('t-SNE of LangChain Embeddings (384-dim --> 2-dim)', fontsize=14, fontweight='bold')
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('Notice how each category forms a distinct cluster!')
print('This is why simple classifiers work so well on embeddings --')
print('the hard work of understanding language was already done by the embedding model.')

## Step 13: Reflection -- Hour 2

| What You Did | What You Learned |
|-------------|------------------|
| Created embeddings with LangChain | How to convert text to meaning vectors |
| Computed cosine similarity | Embeddings capture semantic meaning |
| Implemented 3 chunking strategies | How to split long documents for embedding |
| Compared chunking approaches | Fixed-size vs sentence vs recursive splitting |
| Built a ChromaDB vector store | How vector databases store and search by meaning |
| Ran semantic search queries | Search by meaning, not keywords |
| Trained sklearn classifiers on embeddings | Embeddings + simple ML = powerful pipeline |
| Visualized embedding space with t-SNE | Documents cluster by topic in embedding space |

**Key insight:** LangChain + ChromaDB give you a complete semantic search pipeline in a few lines of code. Choosing the right chunking strategy matters for retrieval quality.

**Think about it:** If you were building a chatbot that answers questions about a 500-page textbook, which chunking strategy would you choose and why?

---

# HOUR 3: Full Pipeline -- DNN vs Embeddings

**Goal:** Compare training a DNN from scratch vs using embeddings + classical ML on the **same task**.

---

## Exercise 3: The Big Comparison

We will compare two approaches on the **20 Newsgroups** text classification task:

1. **DNN from Scratch:** TF-IDF features --> PyTorch neural network --> trained from scratch
2. **Embeddings + ML:** LangChain embeddings --> Logistic Regression --> no training of the embedding model

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

def approach_dnn(texts_train, y_train, texts_test, y_test, n_classes):
    """Approach 1: DNN from scratch using TF-IDF features."""
    t_start = time.time()

    # Step 1: Convert text to TF-IDF features (bag of words, weighted)
    tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
    X_tr = torch.tensor(tfidf.fit_transform(texts_train).toarray(), dtype=torch.float32).to(device)
    X_te = torch.tensor(tfidf.transform(texts_test).toarray(), dtype=torch.float32).to(device)
    y_tr = torch.tensor(y_train, dtype=torch.long).to(device)

    # Step 2: Build a neural network
    dnn = nn.Sequential(
        nn.Linear(5000, 256), nn.ReLU(), nn.Dropout(0.3),
        nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.2),
        nn.Linear(128, n_classes),
    ).to(device)

    # Step 3: Train
    criterion = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(dnn.parameters(), lr=0.001)

    dnn.train()
    for epoch in range(10):
        opt.zero_grad()
        loss = criterion(dnn(X_tr), y_tr)
        loss.backward()
        opt.step()

    # Step 4: Predict
    dnn.eval()
    with torch.no_grad():
        preds = dnn(X_te).argmax(1).cpu().numpy()

    return {
        'name': 'DNN from Scratch',
        'accuracy': accuracy_score(y_test, preds),
        'time': time.time() - t_start
    }


def approach_embeddings(texts_train, y_train, texts_test, y_test, n_classes):
    """Approach 2: Pre-trained embeddings + Logistic Regression."""
    t_start = time.time()

    # Step 1: Embed all texts (using the pre-trained model)
    X_tr = np.array(embeddings_model.embed_documents(texts_train))
    X_te = np.array(embeddings_model.embed_documents(texts_test))

    # Step 2: Train a simple classifier
    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(X_tr, y_train)

    return {
        'name': 'Embeddings + LogReg',
        'accuracy': accuracy_score(y_test, clf.predict(X_te)),
        'time': time.time() - t_start
    }

print('Both approaches defined. Running comparison...')

In [ ]:
print('Running DNN approach...')
result_dnn = approach_dnn(
    train_data.data, train_data.target,
    test_data.data, test_data.target,
    len(categories)
)
print(f'  {result_dnn["name"]}: {result_dnn["accuracy"]:.1%} in {result_dnn["time"]:.1f}s')

print('\nRunning Embeddings approach...')
result_emb = approach_embeddings(
    train_data.data, train_data.target,
    test_data.data, test_data.target,
    len(categories)
)
print(f'  {result_emb["name"]}: {result_emb["accuracy"]:.1%} in {result_emb["time"]:.1f}s')

results = [result_dnn, result_emb]

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
bar_names = [r['name'] for r in results]
bar_colors = [RED, BLUE]

# Accuracy comparison
accs = [r['accuracy'] * 100 for r in results]
bars1 = ax1.bar(bar_names, accs, color=bar_colors, edgecolor='white', linewidth=1.5, width=0.4)
for bar, val in zip(bars1, accs):
    ax1.text(bar.get_x() + bar.get_width() / 2., bar.get_height() + 0.5,
             f'{val:.1f}%', ha='center', fontweight='bold', fontsize=14)
ax1.set_ylabel('Accuracy (%)')
ax1.set_title('Accuracy Comparison', fontsize=14, fontweight='bold')
ax1.set_ylim(0, 105)
ax1.grid(axis='y', alpha=0.3)

# Time comparison
times = [r['time'] for r in results]
bars2 = ax2.bar(bar_names, times, color=bar_colors, edgecolor='white', linewidth=1.5, width=0.4)
for bar, val in zip(bars2, times):
    ax2.text(bar.get_x() + bar.get_width() / 2., bar.get_height() + 0.3,
             f'{val:.1f}s', ha='center', fontweight='bold', fontsize=14)
ax2.set_ylabel('Total Time (seconds)')
ax2.set_title('Time Comparison', fontsize=14, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

plt.suptitle('DNN from Scratch vs Embeddings + ML', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Decision Guide: When to Use Which?

| Factor | Train a DNN from Scratch | Use Embeddings + ML |
|--------|--------------------------|---------------------|
| **Dataset size** | Large (10K+ labeled samples) | Small to medium (even 100 samples) |
| **Domain** | Highly specialized (medical, legal) | General-purpose text |
| **Time budget** | Hours to days | Minutes |
| **Hardware** | GPU recommended | CPU is fine |
| **Interpretability** | Black box | More transparent |
| **Customization** | Full control over architecture | Limited to embedding model's knowledge |
| **Data type** | Any (images, tabular, text) | Text and some structured data |

**Rule of thumb:** Start with embeddings + ML. Only go to training from scratch if you need more accuracy and have the data, time, and compute.

## The Transfer Learning Spectrum

These two approaches sit at opposite ends of a spectrum:

```
Train from Scratch  <-------->  Fine-Tune  <-------->  Use Pre-Trained
     (Hour 1)             (middle ground)              (Hour 2)
  Most effort                                        Least effort
  Most data needed                                   Least data needed
  Most customizable                                  Least customizable
```

**Fine-tuning** (the middle ground) takes a pre-trained model and adjusts it slightly on your specific data. This is how most real-world applications work (e.g., fine-tuning BERT for sentiment analysis).

---

## Workshop Summary

| Hour | What You Built | Key Takeaway |
|------|----------------|---------------|
| **1** | Feedforward DNN (Fashion-MNIST) | The training loop is universal: forward --> loss --> backward --> step |
| **2** | LangChain + ChromaDB pipeline with chunking | Pre-trained models + vector DBs + smart chunking = powerful search & classification |
| **3** | DNN vs Embeddings comparison | Start simple with embeddings, scale up only when needed |

### Tools You Learned

| Tool | What It Does | When to Use |
|------|-------------|-------------|
| **PyTorch** | Train neural networks from scratch | Custom architectures, lots of data |
| **LangChain** | Clean interface to embeddings & LLMs | Any AI pipeline |
| **ChromaDB** | Vector database for semantic search | Search, RAG, recommendations |
| **scikit-learn** | Classical ML classifiers | Quick classification on embeddings |
| **RecursiveCharacterTextSplitter** | Smart document chunking | Splitting long docs for embedding |

### Next Steps

1. **Try the Streamlit app** -- explore embeddings and ChromaDB interactively
2. **Build a RAG system** -- combine ChromaDB with an LLM for question answering
3. **Experiment with chunking** -- try different chunk sizes on your own documents
4. **Fine-tune a model** -- the middle ground between embeddings and DNN from scratch